# 🏗️ Structural Calc Copilot
**Local AI assistant powered by `qwen2.5-coder:7b-instruct` via Ollama**

Knows about: `calc_setup`, `handcalcs`, `forallpeople`, `structuralcodes`, EC2/EC3

---

In [4]:
import requests
from IPython.display import display, Markdown, Code
import ipywidgets as widgets

# ── System prompt baked with your calc_setup knowledge ────────────────────────
SYSTEM_PROMPT = """
You are a structural engineering Python copilot. You help write Jupyter notebook cells
for hand calculations following Eurocode standards (EC2 for concrete, EC3 for steel).

ENVIRONMENT: Every notebook starts with `from calc_setup import *` which provides:

UNITS (via forallpeople 'structural' environment):
  mm, cm, m, km, kg, kN, MN, MPa, GPa, kPa, N, kNm, rad, deg
  Use units on all values e.g. b = 400*mm, fck = 30*MPa

HANDCALCS:
  - Use %%render magic at top of cell to render LaTeX output
  - Use %%render params for parameter-only cells (no equations)
  - Use @handcalc decorator on functions to render them
  - precision is set to 3 decimal places
  - underscore_subscripts=True so var_ck becomes v_{ck} in LaTeX
  - Custom symbols: Lambda_ → Λ, lambda_ → λ

STRUCTURALCODES:
  - set_design_code('ec2_2004') is default (also: 'en_1992_1_1_2023')
  - create_concrete(fck=30) creates a concrete material with EC2 partial factors
  - Access design strength: concrete.fcd (design compressive strength)

SECTIONS DATABASE:
  - db = SectionDB loaded from JSON
  - prefer = ["BSShapes2006", "ArcelorMittal_British", "ArcelorMittal_Europe"]
  - Query with db.find() or db.get()

CODE STYLE RULES:
  1. Always define parameters in a %%render params cell first
  2. Group calculations logically in separate %%render cells
  3. Add markdown cells with section headings between calc cells
  4. Use descriptive variable names with subscripts e.g. M_Ed, V_Ed, A_s
  5. Always check utilisation ratio at the end: e.g. UC = M_Ed / M_Rd
  6. Comment EC2 clause references e.g. # EC2 Cl. 6.1

TYPICAL NOTEBOOK STRUCTURE:
  Cell 1: from calc_setup import *
  Cell 2: ## Project info markdown
  Cell 3: %%render params  (geometry + loads)
  Cell 4: ## Section heading markdown
  Cell 5: %%render  (calculations)
  Cell 6: Utilisation check

When asked to generate notebook cells, output clean Python code blocks.
When asked questions, answer concisely with EC2/EC3 clause references.
Always use SI units consistently throughout.
"""

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5-coder:7b-instruct"

# ── Conversation history (multi-turn) ─────────────────────────────────────────
conversation_history = []

def ask_copilot(user_message, stream=True):
    """Send a message to the local copilot and display the response."""
    conversation_history.append({"role": "user", "content": user_message})
    
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history,
        "stream": stream
    }
    
    print(f"🤖 Copilot thinking...\n")
    
    response = requests.post(OLLAMA_URL, json=payload, stream=stream)
    
    full_response = ""
    if stream:
        import json
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line)
                if not chunk.get("done"):
                    token = chunk["message"]["content"]
                    full_response += token
                    print(token, end="", flush=True)
    else:
        full_response = response.json()["message"]["content"]
        display(Markdown(full_response))
    
    conversation_history.append({"role": "assistant", "content": full_response})
    print("\n\n---")
    return full_response

def reset_conversation():
    """Clear conversation history to start fresh."""
    global conversation_history
    conversation_history = []
    print("✅ Conversation reset.")

def show_history():
    """Show conversation history."""
    for msg in conversation_history:
        role = "👤 You" if msg["role"] == "user" else "🤖 Copilot"
        print(f"{role}: {msg['content'][:200]}...\n")

print("✅ Calc Copilot ready! Use ask_copilot('your question') to start.")
print(f"   Model: {MODEL}")
print(f"   Endpoint: {OLLAMA_URL}")

✅ Calc Copilot ready! Use ask_copilot('your question') to start.
   Model: qwen2.5-coder:7b-instruct
   Endpoint: http://localhost:11434/api/chat


---
## 💬 Chat with your Copilot

Use `ask_copilot("your request")` in any cell below.

**Example prompts:**
- `ask_copilot("Write cells for EC2 rectangular beam moment capacity check")`
- `ask_copilot("Write a shear capacity check for a 400x600 beam with fck=30 MPa")`
- `ask_copilot("What is the EC2 formula for minimum tension reinforcement?")`
- `ask_copilot("Write a column axial + bending interaction check to EC2")`

The copilot **remembers context** within a session. Use `reset_conversation()` to start fresh.

---

In [5]:
# ── Example: Ask for a beam moment capacity check ─────────────────────────────
ask_copilot("Write notebook cells for an EC2 rectangular beam sagging moment capacity check. "
            "Beam is 400mm wide, 600mm deep, fck=30MPa, fyk=500MPa, 4T20 bars, c_nom=30mm.")

🤖 Copilot thinking...

```python
from calc_setup import *
```

## Project info markdown

This notebook performs a sagging moment capacity check for a rectangular beam using Eurocode EC2.

## Geometry and Loads markdown

- Beam dimensions: 
  - Width (b) = 400 mm
  - Depth (d) = 600 mm
- Concrete compressive strength, fck = 30 MPa
- Steel yield strength, fyk = 500 MPa
- Stirrup bars: 4T20 (nominal diameter d_b = 20 mm)
- Nominal cover to the outer face of reinforcing bars, c_nom = 30 mm

## Material Properties markdown

- Concrete:
  - fcd = \( \frac{fck}{1.5} \) # EC2 Cl. 9.1.1
- Steel:
  - fyd = \( \sqrt{\frac{1.15fyk}{1-\gamma_{m,2}} \times \left(1 + \frac{c_{nom} \beta_1}{d}\right)^2} \) # EC2 Cl. 9.1.3
  - γ_m,2 = 0.85 for concrete with fck ≤ 40 MPa

## Section Heading markdown

### Sagging Moment Capacity Check

```python
%%render params
# Geometry and Loads
b = 400 * mm  # Width of the beam
d = 600 * mm  # Depth of the beam
fck = 30 * MPa  # Concrete compressive strength
fyk = 50

'```python\nfrom calc_setup import *\n```\n\n## Project info markdown\n\nThis notebook performs a sagging moment capacity check for a rectangular beam using Eurocode EC2.\n\n## Geometry and Loads markdown\n\n- Beam dimensions: \n  - Width (b) = 400 mm\n  - Depth (d) = 600 mm\n- Concrete compressive strength, fck = 30 MPa\n- Steel yield strength, fyk = 500 MPa\n- Stirrup bars: 4T20 (nominal diameter d_b = 20 mm)\n- Nominal cover to the outer face of reinforcing bars, c_nom = 30 mm\n\n## Material Properties markdown\n\n- Concrete:\n  - fcd = \\( \\frac{fck}{1.5} \\) # EC2 Cl. 9.1.1\n- Steel:\n  - fyd = \\( \\sqrt{\\frac{1.15fyk}{1-\\gamma_{m,2}} \\times \\left(1 + \\frac{c_{nom} \\beta_1}{d}\\right)^2} \\) # EC2 Cl. 9.1.3\n  - γ_m,2 = 0.85 for concrete with fck ≤ 40 MPa\n\n## Section Heading markdown\n\n### Sagging Moment Capacity Check\n\n```python\n%%render params\n# Geometry and Loads\nb = 400 * mm  # Width of the beam\nd = 600 * mm  # Depth of the beam\nfck = 30 * MPa  # Concrete com

In [6]:
# ── Follow-up question (copilot remembers context) ────────────────────────────
# ask_copilot("Now add a shear check for VEd = 200 kN")

In [7]:
# ── Ask an EC2 theory question ─────────────────────────────────────────────────
# ask_copilot("What is the EC2 cl. 6.2.2 formula for concrete shear resistance VRdc?")

In [8]:
# ── Reset and start a new topic ────────────────────────────────────────────────
# reset_conversation()
# ask_copilot("Write cells for an EC3 steel beam moment capacity check, UB 457x191x82")

In [9]:
# ── Free prompt — edit and run ─────────────────────────────────────────────────
# ask_copilot("YOUR QUESTION HERE")